📦 Step 1: Environment Optimization & Model Load

Clear previous memory overhead and pull down google/gemma-2-2b-it natively in low-precision format (bfloat16) to fit comfortably within the 15GB VRAM limits of a free-tier Colab T4 GPU.

In [1]:
!pip install transformer-lens plotly datasets huggingface_hub

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 4.2 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=7978e097e668c7411874c6dc46cb7975bc841831121a3b8c6b619f2eb6a36fc7
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator


In [2]:
# =====================================================================
# STEP 1: INITIALIZE ENVIRONMENT AND DOWNLOAD GEMMA MODEL WEIGHTS
# =====================================================================
import os
import gc
import torch
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM

# Clear VRAM allocations from any previous crashed sessions
torch.cuda.empty_cache()
gc.collect()

# Securely grab your Hugging Face Token from your Colab Secrets drawer
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

print("📥 Step 1: Downloading google/gemma-2-2b-it from Hugging Face Hub...")

# Download and load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")

# Download and instantiate model weights in bfloat16 for optimal T4 compatibility
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2-2b-it",
    torch_dtype=torch.bfloat16,
    device_map="cuda",          # Pulls the download directly into GPU VRAM
    low_cpu_mem_usage=True      # Prevents Colab system RAM crashes
)

print("✅ Step 1 Complete: Model loaded successfully into memory.")


📥 Step 1: Downloading google/gemma-2-2b-it from Hugging Face Hub...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

✅ Step 1 Complete: Model loaded successfully into memory.


🧮 Step 2: Sliced Forward Pass & Activation Hooking

Define your contrastive corporate targets, slice them to the matching min_length constraint, and execute dual forward passes with output_hidden_states=True to take internal snapshots of all 26 layers.

In [11]:
text_clean = "Q3 net profit decreased by 14% due to rising supply chain operational costs."
text_deceptive = "We optimized our structural cost vectors to strategically align long-term shareholder values."

inputs_clean = tokenizer(text_clean, return_tensors="pt").to("cuda")
inputs_deceptive = tokenizer(text_deceptive, return_tensors="pt").to("cuda")

min_length = min(inputs_clean['input_ids'].shape[1], inputs_deceptive['input_ids'].shape[1])
ids_clean = inputs_clean['input_ids'][:, :min_length]
ids_deceptive = inputs_deceptive['input_ids'][:, :min_length]

with torch.no_grad():
    outputs_clean = model(ids_clean, output_hidden_states=True)
    outputs_deceptive = model(ids_deceptive, output_hidden_states=True)

hidden_clean = outputs_clean.hidden_states[1:]
hidden_deceptive = outputs_deceptive.hidden_states[1:]


📐 Step 3: Layer-Isolated Geometric Vector Mathematics

Isolate the absolute last token position vector (target_token_idx = -1) to evaluate the model's exact "thinking boundary." Subtract the clean spatial vector from the deceptive vector, and apply the L2 Euclidean Norm (torch.norm) to calculate the absolute geometric divergence metric across every layer.

In [12]:
clean_magnitudes, deceptive_magnitudes, layer_deviations = [], [], []
target_token_idx = -1

for layer_idx in range(len(hidden_clean)):
    layer_clean = hidden_clean[layer_idx].squeeze(0).detach().cpu().to(torch.float32)
    layer_deceptive = hidden_deceptive[layer_idx].squeeze(0).detach().cpu().to(torch.float32)

    vec_clean = layer_clean[target_token_idx]
    vec_deceptive = layer_deceptive[target_token_idx]

    clean_magnitudes.append(torch.norm(vec_clean, p=2).item())
    deceptive_magnitudes.append(torch.norm(vec_deceptive, p=2).item())
    layer_deviations.append(torch.norm(vec_deceptive - vec_clean, p=2).item())


📊 Step 4: Multi-Track Data Visualization & Data Logs

Convert your extracted tracking metrics into a structured Pandas DataFrame. Render a dual-track line plot to map the divergence paths alongside a text string matrix log formatted cleanly to four decimal points.

In [13]:
import pandas as pd
import plotly.express as px

df_compare = pd.DataFrame({
    "Layer": list(range(len(hidden_clean))) * 2,
    "Activation Norm": clean_magnitudes + deceptive_magnitudes,
    "Prompt Type": ["Honest Audit Run"] * len(hidden_clean) + ["Deceptive Corporate Spin"] * len(hidden_clean)
})
fig_compare = px.line(df_compare, x="Layer", y="Activation Norm", color="Prompt Type", title="Activation Energy Divergence", markers=True)
fig_compare.show()

df_diff = pd.DataFrame({"Layer": list(range(len(layer_deviations))), "Deviation Magnitude": layer_deviations})
print(df_diff.to_string(index=False, formatters={"Deviation Magnitude": "{:.4f}".format}))


 Layer Deviation Magnitude
     0            108.4164
     1            106.9633
     2            104.4574
     3            106.5413
     4            118.2844
     5            122.8990
     6            126.3401
     7            139.5132
     8            148.6844
     9            157.3379
    10            171.3300
    11            189.1686
    12            192.5203
    13            212.8099
    14            226.9519
    15            249.5070
    16            274.8811
    17            282.2695
    18            306.7427
    19            350.3547
    20            378.4507
    21            425.7584
    22            461.9613
    23            515.2681
    24            597.8858
    25            121.2548


# 📑 Note: The complete technical report, methodology analysis, and research write-up are hosted on the main repository landing page here:

https://github.com/SRINIVASTA/MATS_12_Financial_Deception_Probing_Gemma2

In [16]:
# =====================================================================
# COLAB FIX: PACK AND EXPORT ALL 4 DATA COLUMNS
# =====================================================================
from google.colab import files

num_layers = len(hidden_clean)

# Explicitly assign names to all 4 data arrays
df_diff_complete = pd.DataFrame({
    "Layer": list(range(num_layers)),
    "Honest Activation": clean_magnitudes,
    "Deceptive Activation": deceptive_magnitudes,
    "Net Vector Drift (Delta)": layer_deviations
})

# Save it using your exact file name
df_diff_complete.to_csv("layer_metrics.csv", index=False)

print(f"📊 Columns checking: {list(df_diff_complete.columns)}")
print("📥 Downloading updated 4-column metrics log file...")
files.download("layer_metrics.csv")


📊 Columns checking: ['Layer', 'Honest Activation', 'Deceptive Activation', 'Net Vector Drift (Delta)']
📥 Downloading updated 4-column metrics log file...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>